## 1) Configuración rápida

In [1]:
from __future__ import annotations
import pandas as pd
import numpy as np
from pathlib import Path

# === Ajusta aquí si deseas otro archivo o ruta ===
CSV_PATH = Path('../df_procesados/df_planta_2.csv')
# Ventanas y paso
WIN_MIN   = '30min'   # tamaño de ventana móvil
STEP_MIN  = '10min'   # paso entre ventanas

# Baseline: por defecto, primeras 2 semanas del dataset
BASELINE_DAYS = 14

# Semáforo (puedes refinar según tu planta)
PSI_WARN, PSI_ALARM = 0.10, 0.25
KS_P_ALARM = 0.01

# Opcional: forzar nombre de columna datetime si lo conoces
FORCED_DT_COL = None  # e.g., 'date_time'


## 2) Carga de datos y detección automática de columnas (fecha, numéricas y KPI)

In [2]:
def infer_datetime_col(df: pd.DataFrame, forced: str | None = None) -> str:
    if forced and forced in df.columns:
        return forced
    # candidatos por nombre
    name_cands = [c for c in df.columns if str(c).lower() in (
        'date_time', 'datetime', 'timestamp', 'fecha', 'fechahora', 'time')]
    for c in name_cands:
        try:
            pd.to_datetime(df[c])
            return c
        except Exception:
            pass
    # fallback: probar todas las columnas
    for c in df.columns:
        try:
            pd.to_datetime(df[c])
            return c
        except Exception:
            continue
    raise ValueError('No se pudo inferir columna temporal. Especifica FORCED_DT_COL.')

def detect_kpi_cols(cols: list[str]) -> list[str]:
    # heurística: buscar nombres con turbidez/pH clarificado/efluente/kpi
    keys = ['turbidez', 'clarificado', 'kpi', 'efluente', 'salida']
    return [c for c in cols if any(k in str(c).lower() for k in keys)]

raw = pd.read_csv(CSV_PATH)
raw['date_time'] = pd.to_datetime(raw['date_time'])
dt_col = infer_datetime_col(raw, FORCED_DT_COL)
raw[dt_col] = pd.to_datetime(raw[dt_col], errors='coerce')
raw = raw.dropna(subset=[dt_col]).sort_values(dt_col).set_index(dt_col)

num_cols = raw.select_dtypes(include=['number']).columns.tolist()
kpi_cands = detect_kpi_cols(num_cols)
kpi_col = kpi_cands[0] if kpi_cands else None

print('Columna tiempo:', dt_col)
print('Columnas numéricas ({}):'.format(len(num_cols)))
print(num_cols)
print('KPI detectado:', kpi_col)

df = raw.copy()
start, end = df.index.min(), df.index.max()
bl_end = start + pd.Timedelta(days=BASELINE_DAYS)
base = df.loc[start:bl_end]
cur  = df.loc[bl_end:]
print('Baseline:', start, '→', bl_end, '| Actual:', bl_end, '→', end)

Columna tiempo: date_time
Columnas numéricas (12):
['Flujo Cámara de contacto 1', 'Flujo Salida de Lodo Sedimentador Secundario 3', 'Flujo Aire Reactor 2', 'Nivel Pozo WAS', 'Flujo Afluente PTAR', 'Flujo Alimentación Digestor 1', 'Flujo Alimentación Digestor 2', 'Flujo Alimentación Digestor 3', 'Flujo Aire Reactor 1', 'Flujo Aire Reactor 3', 'OD Reactor 1', 'Flujo Cámara de contacto 2']
KPI detectado: Flujo Salida de Lodo Sedimentador Secundario 3
Baseline: 2024-08-12 16:32:00 → 2024-08-26 16:32:00 | Actual: 2024-08-26 16:32:00 → 2025-08-09 23:59:00


## 3) Drift Univariado (Evidently) — PSI/KS/AD y Reporte HTML

In [3]:
from evidently import Report
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift, DriftedColumnsCount

ref = base.copy()
cur = cur.copy()
cols = list(ref.columns)

# Construcción de métricas (preset + univariado + conteo de columnas derivadas)
metrics = [
    DataDriftPreset(),               # preset de drift a nivel dataset
    DriftedColumnsCount(),           # conteo de columnas con drift
    *[ValueDrift(column=c) for c in cols]  # drift univariado por columna
]

report = Report(metrics=metrics)

# Ejecutar reporte
snap = report.run(
    reference_data=ref.reset_index(drop=True),
    current_data=cur.reset_index(drop=True),
)

# Guardar HTML y JSON desde el snapshot (no desde Report)
snap.save_html("planta2_univar.html")
with open("planta2_univar.json", "w", encoding="utf-8") as f:
    f.write(snap.json())            # o json.dump(snap.dict(), f, indent=2, ensure_ascii=False)

print("OK -> planta2_univar.html y planta2_univar.json")


OK -> planta2_univar.html y planta2_univar.json


## 4) Drift Multivariado — MMD (alibi-detect)

In [4]:
%pip install alibi-detect[tensorflow]

from alibi_detect.cd import MMDDrift

x_ref = base[num_cols].dropna().values
mmd = MMDDrift(x_ref, p_val=0.01, kernel='rbf')

def mmd_on_window(df_win):
    X = df_win[num_cols].dropna().values
    if len(X) < 20:
        return {'p_val': 1.0, 'is_drift': 0, 'distance': 0.0}
    res = mmd.predict(X, return_p_val=True, return_distance=True)
    return {
        'p_val': float(res['data']['p_val']),
        'is_drift': int(res['data']['is_drift']),
        'distance': float(res['data']['distance'])
    }

  Using cached tensorflow-2.14.1-cp310-cp310-win_amd64.whl (2.1 kB)
  Using cached tensorflow_intel-2.14.1-cp310-cp310-win_amd64.whl (284.1 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\frncc\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python310\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\ext\\filters\\client_channel\\lb_policy\\grpclb\\client_load_reporting_filter.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: C:\Users\frncc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: `tensorflow` not installed. Cannot initialize and run MMDDrift with tensorflow backend. The necessary missing dependencies can be installed using `pip install alibi-detect[tensorflow]`.

## 5) PCA → Hotelling T² y Q (SPE)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Xb = base[num_cols].dropna()
scaler = StandardScaler().fit(Xb)
Zb = scaler.transform(Xb)
pca = PCA().fit(Zb)

def t2_q_window(df_win):
    Xw = df_win[num_cols].dropna()
    if Xw.empty:
        return np.nan, np.nan
    Zw = scaler.transform(Xw)
    scores = pca.transform(Zw)
    var = np.var(scores, axis=0, ddof=1)
    var[var == 0] = 1e-9
    T2_inst = np.sum((scores**2) / var, axis=1)
    T2_mean = float(np.mean(T2_inst))
    Zw_rec = pca.inverse_transform(scores)
    resid = Zw - Zw_rec
    Q = float(np.mean(np.sum(resid**2, axis=1)))
    return T2_mean, Q

## 6) KPI / Concept Drift — Page–Hinkley (si existe KPI detectado)

In [ ]:
from river.drift import PageHinkley

if kpi_col is not None:
    ph = PageHinkley(min_instances=30, delta=0.005, threshold=50.0, alpha=0.9999)
    flags = []
    idxs = []
    for ts, v in cur[kpi_col].dropna().items():
        ph.update(v)
        flags.append(ph.change_detected)
        idxs.append(ts)
        if ph.change_detected_:
            ph.reset()
    kpi_flags = pd.Series(flags, index=idxs, dtype=bool)
    kpi_flags.to_frame('PH_change').to_csv('planta1_kpi_ph.csv')
    print('Guardado: planta1_kpi_ph.csv')
else:
    print('No se detectó KPI; se omite Page–Hinkley.')

AttributeError: 'PageHinkley' object has no attribute 'change_detected_'

## 7) Anomalías / Nuevos Regímenes — IsolationForest + LOF (PyOD)

In [ ]:
from sklearn.ensemble import IsolationForest
from pyod.models.lof import LOF

Xb_np = base[num_cols].dropna().values
iforest = IsolationForest(contamination=0.02, random_state=0).fit(Xb_np)
lof = LOF(contamination=0.02).fit(Xb_np)

def outlier_ratios(df_win):
    X = df_win[num_cols].dropna().values
    if len(X) == 0:
        return 0.0, 0.0
    r_if = (iforest.predict(X) == -1).mean()
    r_lof = lof.predict(X).mean()  # 1=outlier
    return float(r_if), float(r_lof)

## 8) Bucle por ventanas + Semáforo + Export a CSV

In [ ]:
def sliding_windows(df, win=WIN_MIN, step=STEP_MIN):
    start = df.index.min()
    end = df.index.max()
    t = start
    while t + pd.Timedelta(win) <= end:
        w = df.loc[t: t + pd.Timedelta(win)]
        yield (t, t + pd.Timedelta(win), w)
        t = t + pd.Timedelta(step)

rows = []
for t0, t1, w in sliding_windows(cur):
    mmd_res = mmd_on_window(w)
    T2, Q = t2_q_window(w)
    r_if, r_lof = outlier_ratios(w)

    # Semáforo simple (ajústalo a tu operación):
    level = 'Verde'
    if mmd_res['is_drift'] == 1 or r_if > 0.10 or r_lof > 0.10:
        level = 'Amarillo'
    if (mmd_res['is_drift'] == 1 and (r_if > 0.10 or r_lof > 0.10)) or (not np.isnan(T2) and T2 > 10):
        level = 'Rojo'

    rows.append({
        't_ini': t0, 't_fin': t1,
        'MMD_p': mmd_res['p_val'], 'MMD_is_drift': mmd_res['is_drift'], 'MMD_dist': mmd_res['distance'],
        'T2_mean': T2, 'Q_mean': Q,
        'outlier_iforest': r_if, 'outlier_lof': r_lof,
        'semaforo': level
    })

summary = pd.DataFrame(rows)
summary.to_csv('planta1_ventanas.csv', index=False)
summary.head()

NameError: name 'mmd_on_window' is not defined

## 9) Visualizaciones rápidas (opcional)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12,4))
plt.plot(summary['t_fin'], summary['MMD_dist'])
plt.title('Distancia MMD por ventana')
plt.xlabel('Tiempo')
plt.ylabel('MMD distance')
plt.show()

fig = plt.figure(figsize=(12,4))
plt.plot(summary['t_fin'], summary['T2_mean'])
plt.title('T² medio por ventana')
plt.xlabel('Tiempo')
plt.ylabel('T² (promedio)')
plt.show()

fig = plt.figure(figsize=(12,4))
plt.plot(summary['t_fin'], summary['outlier_iforest'], label='IForest')
plt.plot(summary['t_fin'], summary['outlier_lof'], label='LOF')
plt.title('Proporción de outliers por ventana')
plt.xlabel('Tiempo')
plt.ylabel('Ratio outliers')
plt.legend()
plt.show()

fig = plt.figure(figsize=(12,2.5))
y = summary['semaforo'].map({'Verde':0, 'Amarillo':1, 'Rojo':2}).values
plt.step(summary['t_fin'], y, where='post')
plt.yticks([0,1,2], ['Verde','Amarillo','Rojo'])
plt.title('Semáforo por ventana')
plt.xlabel('Tiempo')
plt.show()

NameError: name 'summary' is not defined

<Figure size 1200x400 with 0 Axes>

### Notas y buenas prácticas
- Revisa `num_cols` y `kpi_col` detectados; si necesitas fijarlos manualmente, edítalos.
- Ajusta `BASELINE_DAYS`, `WIN_MIN`, `STEP_MIN` y umbrales del semáforo.
- Usa `planta1_univar.html` para ranking de columnas con mayor drift (PSI/KS/AD).
- Si el KPI **cambia** sin que MMD marque data drift, investiga **concept drift** (química/setpoints/modelo).